In [5]:
# Cell 0: Config
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import norm
import scipy.stats as stats
import numpy as np
from linearmodels.panel import PanelOLS

SPREADS_PATH = '../output/Current_results/complete_results_weekly_15_dampening.csv'
DATE_COL     = 'date'
COUNTRY_COL  = 'country'
CDS_COL      = 'cds_spread'
YIELD_COL    = 'yield_market'
HORIZON = 5
RECOVERY = 0.4
HORIZONS     = [1, 2, 4, 8]
EXPORTERS    = ['Saudi Arabia', 'UAE (Abu Dhabi)', 'Colombia',
                'Mexico', 'Brazil', 'Egypt', 'Malaysia', 'Qatar']
CONTROLS     = ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa',
                'South Korea', 'Thailand', 'Turkey']

panel = pd.read_csv(SPREADS_PATH, parse_dates=[DATE_COL])

In [6]:
# Cell 2: Load controls and merge onto panel

# ── Load macro controls ───────────────────────────────────────
macro = pd.read_csv(
    '../data/processed/Macroeconomic_variables/macro_risk_variables.csv',
    sep=',', dayfirst=True, parse_dates=['Date'], index_col='Date'
)
vix = pd.read_csv(
    '../data/processed/Macroeconomic_variables/VIXCLS.csv',
    parse_dates=['Date'], index_col='Date'
)
ovx = pd.read_csv(
    '../data/processed/Macroeconomic_variables/OVXCLS.csv',
    parse_dates=['date'], index_col='date'
)
gpr = pd.read_csv(
    '../data/processed/Macroeconomic_variables/geopolitical_risk_index_daily.csv',
    sep=';', dayfirst=True, parse_dates=['date'], index_col='date',
    decimal=','
)
for col in gpr.columns:
    gpr[col] = pd.to_numeric(gpr[col], errors='coerce')

oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv',
                           parse_dates=['date'], index_col='date')
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv',
                           dayfirst=True, parse_dates=['date'], index_col='date')

# ── Build controls on daily index ────────────────────────────
macro['VIX']   = vix['VIXCLS']
macro['OVX']   = ovx['OVXCLS']
macro          = macro.join(gpr[['GPRD']], how='left')
macro['basis'] = oil_prices['Brent'] / oil_futures['Brent_12m']
controls_daily = macro[['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis']].copy()

# ── Load FX rates ─────────────────────────────────────────────
fx_wide = pd.read_csv('../data/processed/CCA_V2/exchange_rates.csv',
                       dayfirst=True, parse_dates=['date'])
fx = fx_wide.melt(id_vars='date', var_name=COUNTRY_COL, value_name='fx_rate')
fx = fx.dropna(subset=['fx_rate'])

fx[COUNTRY_COL] = fx[COUNTRY_COL].replace({'United Arab Emirates': 'UAE (Abu Dhabi)'})


# ── Anchor to panel dates ─────────────────────────────────────
cds_dates = pd.DatetimeIndex(panel[DATE_COL].unique())

# ── Reindex controls to panel dates ──────────────────────────
controls_weekly = controls_daily\
    .reindex(cds_dates, method='nearest',
             tolerance=pd.Timedelta('7 days'))\
    .ffill().bfill()\
    .reset_index()\
    .rename(columns={'Date': DATE_COL, 'index': DATE_COL})
controls_weekly[DATE_COL] = pd.to_datetime(controls_weekly[DATE_COL])

# ── Reindex FX per country to panel dates ────────────────────
fx_weekly = pd.merge_asof(
    panel[[DATE_COL, COUNTRY_COL]].sort_values(DATE_COL),
    fx.sort_values(DATE_COL),
    on=DATE_COL,
    by=COUNTRY_COL,
    direction='nearest',
    tolerance=pd.Timedelta('7 days')
)

fx_weekly

# ── Merge controls onto panel ─────────────────────────────────
panel = pd.merge_asof(
    panel.sort_values(DATE_COL),
    controls_weekly.sort_values(DATE_COL),
    on=DATE_COL,
    direction='nearest',
    tolerance=pd.Timedelta('7 days')
)

panel = panel.merge(
    fx_weekly[[DATE_COL, COUNTRY_COL, 'fx_rate']],
    on=[DATE_COL, COUNTRY_COL],
    how='left'
)

panel = panel.sort_values([COUNTRY_COL, DATE_COL]).reset_index(drop=True)

print("Panel shape:", panel.shape)
print("Columns:", panel.columns.tolist())
print("Nulls:\n", panel.isnull().sum()[panel.isnull().sum() > 0])


Panel shape: (8352, 33)
Columns: ['date', 'country', 'cds_spread', 'risk_free_rate', 'B_f', 'LCL_usd', 'sigma_lcl', 'implied_V_M0', 'implied_sigma_V_M0', 'cca_converged_M0', 'implied_V_M1', 'implied_sigma_V_M1', 'cca_converged_M1', 'convenience_yield', 'implied_V_M2', 'implied_sigma_V_M2', 'cca_converged_M2', 'lambda_annual', 'group', 'exporter', 'DD_M0', 'DD_M1', 'DD_M2', 'PD_M0', 'PD_M1', 'PD_M2', 'VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis', 'fx_rate']
Nulls:
 Series([], dtype: int64)


/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_75975/879418672.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv',


# 0: Correlations

In [7]:
# Correlation of levels: CDS vs DD across models
from scipy.stats import pearsonr, spearmanr

print(f"{'Country':<20} | {'M0 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8} | {'M1 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8} | {'M2 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8}")
print("-" * 130)

corrs = {f'{m}_{g}_{t}': [] for m in ['M0','M1','M2'] for g in ['exp','ctl'] for t in ['pear','spear']}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    sub = panel[panel[COUNTRY_COL] == c].copy()
    
    line = f"{c:<20}"
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        tmp = sub[[CDS_COL, dd_col]].dropna()
        tmp = tmp[tmp[CDS_COL] > 0]
        
        if len(tmp) > 10:
            pr, pp = pearsonr(tmp[CDS_COL], tmp[dd_col])
            sr, sp = spearmanr(tmp[CDS_COL], tmp[dd_col])
        else:
            pr, pp, sr, sp = np.nan, np.nan, np.nan, np.nan
        
        corrs[f'{model}_{grp}_pear'].append(pr)
        corrs[f'{model}_{grp}_spear'].append(sr)
        
        p_star = '***' if pp < 0.01 else '**' if pp < 0.05 else '*' if pp < 0.1 else ''
        s_star = '***' if sp < 0.01 else '**' if sp < 0.05 else '*' if sp < 0.1 else ''
        line += f" | {pr:>9.3f}{p_star:<3} {pp:>8.4f} {sr:>9.3f}{s_star:<3} {sp:>8.4f}"
    
    print(line)

print("-" * 130)
for g, label in [('exp', 'Exporters'), ('ctl', 'Controls')]:
    line = f"Mean {label:<15}"
    for m in ['M0', 'M1', 'M2']:
        mp = np.nanmean(corrs[f'{m}_{g}_pear'])
        ms = np.nanmean(corrs[f'{m}_{g}_spear'])
        line += f" | {mp:>10.3f} {'':>8} {ms:>10.3f} {'':>8}"
    print(line)

Country              | M0 Pearson        p   Spearman        p | M1 Pearson        p   Spearman        p | M2 Pearson        p   Spearman        p
----------------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |    -0.605***   0.0000    -0.738***   0.0000 |    -0.654***   0.0000    -0.795***   0.0000 |    -0.680***   0.0000    -0.770***   0.0000
UAE (Abu Dhabi)      |     0.155***   0.0004     0.123***   0.0049 |    -0.369***   0.0000    -0.308***   0.0000 |    -0.284***   0.0000    -0.331***   0.0000
Colombia             |    -0.546***   0.0000    -0.558***   0.0000 |    -0.470***   0.0000    -0.520***   0.0000 |    -0.572***   0.0000    -0.599***   0.0000
Mexico               |    -0.379***   0.0000    -0.379***   0.0000 |    -0.512***   0.0000    -0.471***   0.0000 |    -0.520***   0.0000    -0.484***   0.0000
Brazil               |    -0.524***   0.0000    -0.532***   0.0000 |    -0.564***   0.

# 1. Inseparability Test: OVX and Futures Basis vs Global Risk Factors

In [8]:
# Use unique dates from panel — one row per date for time series regressions
ts = panel.drop_duplicates(subset=[DATE_COL]).set_index(DATE_COL)\
          [['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis']].sort_index()

# ── OVX ~ Global factors (levels) ────────────────────────────
idx   = ts[['OVX', 'VIX', 'DXY', 'UST10Y', 'GPRD']].dropna().index
y_ovx = ts.loc[idx, 'OVX']
X_ovx = sm.add_constant(ts.loc[idx, ['VIX', 'DXY', 'UST10Y', 'GPRD']])
res_ovx = sm.OLS(y_ovx, X_ovx).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

print("OVX ~ Global Factors (levels)")
print(f"R²: {res_ovx.rsquared:.4f}  |  Adj. R²: {res_ovx.rsquared_adj:.4f}")
print(f"\n{'Variable':<12} {'Beta':>10} {'p-value':>10}")
print("-" * 35)
for var in X_ovx.columns:
    print(f"{var:<12} {res_ovx.params[var]:>10.4f} {res_ovx.pvalues[var]:>10.4f}")

OVX ~ Global Factors (levels)
R²: 0.5903  |  Adj. R²: 0.5872

Variable           Beta    p-value
-----------------------------------
const         -101.0461     0.0000
VIX              1.4614     0.0000
DXY              1.3520     0.0000
UST10Y          -7.6438     0.0000
GPRD             0.0104     0.4554


In [9]:
# Convenience yield ~ Global Factors
cy = panel.drop_duplicates(subset=[DATE_COL]).set_index(DATE_COL)[['convenience_yield', 'VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD']].sort_index().dropna()

y = cy['convenience_yield']
X = sm.add_constant(cy[['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD']])
res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

print("Convenience Yield ~ Global Factors (levels)")
print(f"R²: {res.rsquared:.4f}  |  Adj. R²: {res.rsquared_adj:.4f}")
print(f"\n{'Variable':<15} {'Beta':>10} {'p-value':>10}")
print("-" * 38)
for var in X.columns:
    print(f"{var:<15} {res.params[var]:>10.4f} {res.pvalues[var]:>10.4f}")

Convenience Yield ~ Global Factors (levels)
R²: 0.5105  |  Adj. R²: 0.5057

Variable              Beta    p-value
--------------------------------------
const              -0.2401     0.0180
VIX                 0.0056     0.0000
OVX                -0.0041     0.0000
DXY                 0.0027     0.0302
UST10Y              0.0208     0.0004
GPRD                0.0002     0.0449


# 2. Regressions

In [10]:
# ============================================================
# Regression 1: ln(CDS) = α + β·ln(V) + ε  — Asset Value (M0 vs M1)
# ============================================================

print("Regression 1: ln(CDS) = α + β·ln(V) + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 90)

r2_v = {'M0_exp': [], 'M1_exp': [], 'M0_ctl': [], 'M1_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"

    for model, v_col in [('M0', 'implied_V_M0'), ('M1', 'implied_V_M1')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, v_col]].dropna()
        df = df[(df[CDS_COL] > 0) & (df[v_col] > 0)]
        y = np.log(df[CDS_COL])
        X = sm.add_constant(np.log(df[v_col]))
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params.iloc[1]
        p = res.pvalues.iloc[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_v[f'{model}_{grp}'].append(res.rsquared)

    print(line)

print("-" * 90)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_v['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_v['M1_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_v['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_v['M1_ctl']):>7.3f}")

Regression 1: ln(CDS) = α + β·ln(V) + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N
------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.575***  0.0000   0.484   522 |  -0.539***  0.0000   0.532   522
UAE (Abu Dhabi)      |  -0.613***  0.0000   0.389   522 |  -0.553***  0.0000   0.444   522
Colombia             |   0.384*    0.0597   0.043   522 |   0.202     0.2208   0.020   522
Mexico               |  -0.861***  0.0000   0.188   522 |  -0.709***  0.0000   0.275   522
Brazil               |  -3.559***  0.0000   0.500   522 |  -1.659***  0.0000   0.435   522
Egypt                |   1.167***  0.0000   0.279   522 |   1.021***  0.0000   0.289   522
Malaysia             |  -2.616***  0.0000   0.519   522 |  -1.920***  0.0000   0.560   522
Qatar                |  -0.814***  0.0000   0.573   522 |  -0.726***  0.0000   0.594   522
Chile                |  -0.248     0.2492   0.018  

In [11]:
# ============================================================
# Regression 2: ln(CDS) = α + β·ln(σ) + ε  — Volatility (M0 vs M2)
# ============================================================
 
print("\n\nRegression 2: ln(CDS) = α + β·ln(σ) + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 90)
 
r2_vol = {'M0_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M2_ctl': []}
 
for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
 
    for model, sig_col in [('M0', 'implied_sigma_V_M0'), ('M2', 'implied_sigma_V_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, sig_col]].dropna()
        df = df[(df[CDS_COL] > 0) & (df[sig_col] > 0)]
        y = np.log(df[CDS_COL])
        X = sm.add_constant(np.log(df[sig_col]))
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params.iloc[1]
        p = res.pvalues.iloc[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_vol[f'{model}_{grp}'].append(res.rsquared)
 
    print(line)
 
print("-" * 90)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_vol['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_vol['M2_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_vol['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_vol['M2_ctl']):>7.3f}")



Regression 2: ln(CDS) = α + β·ln(σ) + ε

Country              |     M0 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------
Saudi Arabia         |   0.416***  0.0000   0.577   522 |   0.614***  0.0000   0.642   522
UAE (Abu Dhabi)      |  -0.100**   0.0483   0.055   522 |  -0.061     0.5670   0.003   522
Colombia             |   0.622***  0.0020   0.193   522 |   0.763***  0.0011   0.215   522
Mexico               |   0.047     0.7228   0.002   522 |   0.334     0.1112   0.047   522
Brazil               |   0.694***  0.0005   0.168   522 |   0.986***  0.0001   0.236   522
Egypt                |   0.153**   0.0446   0.040   522 |   0.164*    0.0677   0.034   522
Malaysia             |   0.397*    0.0702   0.042   522 |   1.244***  0.0029   0.125   522
Qatar                |   0.403***  0.0000   0.504   522 |   0.644***  0.0000   0.573   522
Chile                |   0.131     0.3148   0.013

In [12]:
print("Regression: ln(CDS) = α + β·DD + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 120)

r2 = {'M0_exp': [], 'M1_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M1_ctl': [], 'M2_ctl': []}
betas = {'M0_exp': [], 'M1_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M1_ctl': [], 'M2_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
    
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, dd_col]].dropna()
        df = df[df[CDS_COL] > 0]
        y = np.log(df[CDS_COL].values)
        X = sm.add_constant(df[dd_col].values)
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        
        b = res.params[1]
        p = res.pvalues[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        
        r2[f'{model}_{grp}'].append(res.rsquared)
        betas[f'{model}_{grp}'].append(b)
    
    print(line)

print("-" * 120)

print(
    f"{'Mean β Exporters':<20} | "
    f"{np.mean(betas['M0_exp']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M1_exp']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M2_exp']):>7.3f}"
)
print(
    f"{'Mean β Controls':<20} | "
    f"{np.mean(betas['M0_ctl']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M1_ctl']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M2_ctl']):>7.3f}"
)
print(
    f"{'Mean R² Exporters':<20} | "
    f"{'':>8} {'':>8} {np.mean(r2['M0_exp']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M1_exp']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M2_exp']):>7.3f}"
)
print(
    f"{'Mean R² Controls':<20} | "
    f"{'':>8} {'':>8} {np.mean(r2['M0_ctl']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M1_ctl']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M2_ctl']):>7.3f}"
)

Regression: ln(CDS) = α + β·DD + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.110***  0.0000   0.448   522 |  -0.121***  0.0000   0.517   522 |  -0.278***  0.0000   0.538   522
UAE (Abu Dhabi)      |   0.009     0.3787   0.014   522 |  -0.053***  0.0061   0.123   522 |  -0.172***  0.0096   0.090   522
Colombia             |  -0.238***  0.0000   0.309   522 |  -0.177***  0.0001   0.240   522 |  -0.346***  0.0000   0.328   522
Mexico               |  -0.111***  0.0090   0.139   522 |  -0.095***  0.0005   0.243   522 |  -0.223***  0.0002   0.255   522
Brazil               |  -0.202***  0.0000   0.297   522 |  -0.164***  0.0000   0.333   522 |  -0.306***  0.0000   0.367   522
Egypt                |  -0.074***  0.0000   0.214   522 |  -0.071***  0.0000   0.181   522

In [13]:
print("Regression: ln(CDS) = α + β·DD + ln(VIX) + ln(UST10Y) + ln(DXY) + ln(GPRD) + ln(FX) + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 120)
r2_c = {'M0_exp': [], 'M1_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M1_ctl': [], 'M2_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
    
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, dd_col, 'VIX', 'UST10Y', 'DXY', 'GPRD', 'fx_rate']].dropna()
        df = df[(df[CDS_COL] > 0) & (df['VIX'] > 0) & (df['UST10Y'] > 0) & (df['DXY'] > 0) & (df['GPRD'] > 0) & (df['fx_rate'] > 0)].reset_index(drop=True)
        
        if len(df) < 30:
            line += f" | {'--':>7}    {'--':>7} {'--':>7} {len(df):>5}"
            continue
        
        y = np.log(df[CDS_COL].values)
        X = np.column_stack([
            np.ones(len(df)),
            df[dd_col].values,
            np.log(df['VIX'].values),
            np.log(df['UST10Y'].values),
            np.log(df['DXY'].values),
            np.log(df['GPRD'].values),
            np.log(df['fx_rate'].values)
        ])
        
        if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
            line += f" | {'--':>7}    {'--':>7} {'--':>7} {len(df):>5}"
            continue
        
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params[1]
        p = res.pvalues[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_c[f'{model}_{grp}'].append(res.rsquared)
    
    print(line)

print("-" * 120)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_c['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M1_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M2_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_c['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M1_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_c['M2_ctl']):>7.3f}")

Regression: ln(CDS) = α + β·DD + ln(VIX) + ln(UST10Y) + ln(DXY) + ln(GPRD) + ln(FX) + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.118***  0.0000   0.496   522 |  -0.125***  0.0000   0.539   522 |  -0.277***  0.0000   0.552   522
UAE (Abu Dhabi)      |   0.003     0.8160   0.102   522 |  -0.050*    0.0942   0.137   522 |  -0.131*    0.0927   0.140   522
Colombia             |  -0.201***  0.0000   0.609   522 |  -0.188***  0.0000   0.643   522 |  -0.334***  0.0000   0.627   522
Mexico               |  -0.120***  0.0022   0.332   522 |  -0.129***  0.0000   0.466   522 |  -0.357***  0.0000   0.510   522
Brazil               |  -0.249***  0.0000   0.492   522 |  -0.236***  0.0000   0.538   522 |  -0.394***  0.0000   0.545   522
Egypt                |  -0.024*    0.

## Pooled Regression

In [14]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


MODEL_MAP = {
    'M0': 'DD_M0',
    'M1': 'DD_M1',
    'M2': 'DD_M2'
}

print("Pooled FE regression: ln(CDS) = α_i + λ_t + β·DD + γ·(DD×Exporter) + ε\n")
print(f"{'Model':<6} | {'β control':>10} {'p':>8} | {'γ exporter diff':>16} {'p':>8} | {'β exporter':>11} {'p':>8} | {'R²':>7} {'N':>6}")
print("-" * 95)

for model_name, dd_col in MODEL_MAP.items():
    df = panel[[COUNTRY_COL, DATE_COL, CDS_COL, dd_col]].dropna().copy()
    df = df[df[CDS_COL] > 0].copy()

    df['ln_cds'] = np.log(df[CDS_COL])
    df['exporter'] = df[COUNTRY_COL].isin(EXPORTERS).astype(int)
    df['dd_exporter'] = df[dd_col] * df['exporter']

    formula = f"ln_cds ~ {dd_col} + dd_exporter + C({COUNTRY_COL}) + C({DATE_COL})"
    res = smf.ols(formula, data=df).fit(
        cov_type='HAC',
        cov_kwds={'maxlags': 12}
    )

    beta = res.params[dd_col]
    p_beta = res.pvalues[dd_col]

    gamma = res.params['dd_exporter']
    p_gamma = res.pvalues['dd_exporter']

    beta_exporter = beta + gamma
    p_exporter = float(res.t_test(f"{dd_col} + dd_exporter = 0").pvalue)

    print(
        f"{model_name:<6} | "
        f"{beta:>10.4f} {p_beta:>8.4f} | "
        f"{gamma:>16.4f} {p_gamma:>8.4f} | "
        f"{beta_exporter:>11.4f} {p_exporter:>8.4f} | "
        f"{res.rsquared:>7.3f} {int(res.nobs):>6}"
    )

Pooled FE regression: ln(CDS) = α_i + λ_t + β·DD + γ·(DD×Exporter) + ε

Model  |  β control        p |  γ exporter diff        p |  β exporter        p |      R²      N
-----------------------------------------------------------------------------------------------
M0     |    -0.0153   0.1297 |          -0.0391   0.0049 |     -0.0544   0.0000 |   0.899   8352
M1     |    -0.0419   0.0000 |          -0.0487   0.0000 |     -0.0906   0.0000 |   0.907   8352
M2     |    -0.0698   0.0010 |          -0.0786   0.0063 |     -0.1485   0.0000 |   0.906   8352


In [17]:
from linearmodels.panel import PanelOLS

for model_name, dd_col in MODEL_MAP.items():
    df = panel[[COUNTRY_COL, DATE_COL, CDS_COL, dd_col]].dropna().copy()
    df = df[df[CDS_COL] > 0].copy()
    df['ln_cds'] = np.log(df[CDS_COL])
    df['exporter'] = df[COUNTRY_COL].isin(EXPORTERS).astype(int)
    df['dd_exporter'] = df[dd_col] * df['exporter']
    df = df.set_index([COUNTRY_COL, DATE_COL])

    mod = PanelOLS(df['ln_cds'], df[[dd_col, 'dd_exporter']], entity_effects=True, time_effects=True)
    res = mod.fit(cov_type='kernel')
    print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                 ln_cds   R-squared:                        0.0967
Estimator:                   PanelOLS   R-squared (Between):             -0.0712
No. Observations:                8352   R-squared (Within):               0.0609
Date:                Sun, Apr 12 2026   R-squared (Overall):             -0.0704
Time:                        09:27:04   Log-likelihood                   -257.37
Cov. Estimator:        Driscoll-Kraay                                           
                                        F-statistic:                      418.11
Entities:                          16   P-value                           0.0000
Avg Obs:                       522.00   Distribution:                  F(2,7813)
Min Obs:                       522.00                                           
Max Obs:                       522.00   F-statistic (robust):             24.170
                            

## Main Specification: Stacked Panel Regression

For each model extension $M_k \in \{M1, M2\}$. We estimate:

$$\ln(\text{CDS}_{it}) = \alpha_i + \lambda_t + \beta_1 \, DD_{it} + \beta_2 \, (DD_{it} \times \text{Exp}_i) + \beta_3 \, (DD_{it} \times \text{Model}_{it}) + \beta_4 \, (DD_{it} \times \text{Exp}_i \times \text{Model}_{it}) + \varepsilon_{it}$$

with entity and time fixed effects. The coefficients map directly to the three hypotheses:
- **$\beta_1$ (Baseline performance in controls):** How well does DD predict CDS for controls under M0.
- **$\beta_2$ (Baseline differential in exporters):** 
- **$\beta_3$ (H1/H2 for controls):** Does the extension improve DD–CDS tracking for control countries?
- **$\beta_4$ (H3):** Does the improvement differ for exporters relative to controls?
- **$\beta_3 + \beta_4$ (H1/H2 for exporters):** Does the extension improve DD–CDS tracking for exporters? Tested via Wald test of $\beta_3 + \beta_4 = 0$.

Standard errors are Driscoll-Kraay (Bartlett kernel). We report results across bandwidths of 8 (≈2 months), 26 (≈6 months), and 52 (≈1 year) to verify robustness to serial dependence assumptions.

In [50]:
# ============================================================
# H3 Test: Does the exporter differential CHANGE across models?
# Triple interaction: DD × Exporter × Model
# ============================================================

from linearmodels.panel import PanelOLS

comparisons = [('M0', 'M1', 'DD_M0', 'DD_M1'),
               ('M0', 'M2', 'DD_M0', 'DD_M2')]

for base_name, ext_name, base_col, ext_col in comparisons:
    # Stack base and extended model
    d0 = panel[[COUNTRY_COL, DATE_COL, CDS_COL, base_col]].dropna().copy()
    d0 = d0[d0[CDS_COL] > 0].copy()
    d0['DD'] = d0[base_col]
    d0['model'] = 0

    d1 = panel[[COUNTRY_COL, DATE_COL, CDS_COL, ext_col]].dropna().copy()
    d1 = d1[d1[CDS_COL] > 0].copy()
    d1['DD'] = d1[ext_col]
    d1['model'] = 1

    stacked = pd.concat([d0, d1], ignore_index=True)
    stacked['ln_cds']       = np.log(stacked[CDS_COL])
    stacked['exporter']     = stacked[COUNTRY_COL].isin(EXPORTERS).astype(int)
    stacked['dd_exp']       = stacked['DD'] * stacked['exporter']
    stacked['dd_model']     = stacked['DD'] * stacked['model']
    stacked['dd_exp_model'] = stacked['DD'] * stacked['exporter'] * stacked['model']

    stacked = stacked.set_index([COUNTRY_COL, DATE_COL])

    X = stacked[['DD', 'dd_exp', 'dd_model', 'dd_exp_model']]
    y = stacked['ln_cds']

    mod = PanelOLS(y, X, entity_effects=True, time_effects=True, drop_absorbed=True)
    res = mod.fit(cov_type='kernel', kernel='bartlett', bandwidth=52)
    #res = mod.fit(cov_type='clustered', cluster_time=True)
    #res = mod.fit(cov_type='clustered', cluster_entity=True)


    g3 = res.params['dd_exp_model']
    p3 = res.pvalues['dd_exp_model']
    sig = '***' if p3 < 0.01 else ('**' if p3 < 0.05 else ('*' if p3 < 0.1 else ''))

    print(f"\n{base_name} → {ext_name}: dd_exp_model = {g3:.4f}  (p = {p3:.4f}) {sig}")
    print(res.summary)
    print(res.wald_test(formula='dd_model + dd_exp_model = 0'))


M0 → M1: dd_exp_model = -0.0087  (p = 0.0567) *
                          PanelOLS Estimation Summary                           
Dep. Variable:                 ln_cds   R-squared:                        0.1133
Estimator:                   PanelOLS   R-squared (Between):             -0.0812
No. Observations:               16704   R-squared (Within):               0.0941
Date:                Sun, Apr 12 2026   R-squared (Overall):             -0.0802
Time:                        11:38:05   Log-likelihood                   -359.48
Cov. Estimator:        Driscoll-Kraay                                           
                                        F-statistic:                      516.41
Entities:                          16   P-value                           0.0000
Avg Obs:                       1044.0   Distribution:                 F(4,16163)
Min Obs:                       1044.0                                           
Max Obs:                       1044.0   F-statistic (robust)

### Same as above but in a reduced summary form

In [51]:
# ============================================================
# Main Specification: Stacked Triple Interaction
# ============================================================
from linearmodels.panel import PanelOLS

comparisons = [('M0', 'M1', 'DD_M0', 'DD_M1'),
               ('M0', 'M2', 'DD_M0', 'DD_M2')]
BANDWIDTHS = [8, 26, 52]

for base_name, ext_name, base_col, ext_col in comparisons:

    d0 = panel[[COUNTRY_COL, DATE_COL, CDS_COL, base_col]].dropna().copy()
    d0 = d0[d0[CDS_COL] > 0].copy()
    d0['DD'] = d0[base_col]
    d0['model'] = 0

    d1 = panel[[COUNTRY_COL, DATE_COL, CDS_COL, ext_col]].dropna().copy()
    d1 = d1[d1[CDS_COL] > 0].copy()
    d1['DD'] = d1[ext_col]
    d1['model'] = 1

    stacked = pd.concat([d0, d1], ignore_index=True)
    stacked['ln_cds']       = np.log(stacked[CDS_COL])
    stacked['exporter']     = stacked[COUNTRY_COL].isin(EXPORTERS).astype(int)
    stacked['dd_exp']       = stacked['DD'] * stacked['exporter']
    stacked['dd_model']     = stacked['DD'] * stacked['model']
    stacked['dd_exp_model'] = stacked['DD'] * stacked['exporter'] * stacked['model']
    stacked = stacked.set_index([COUNTRY_COL, DATE_COL])

    X = stacked[['DD', 'dd_exp', 'dd_model', 'dd_exp_model']]
    y = stacked['ln_cds']
    mod = PanelOLS(y, X, entity_effects=True, time_effects=True, drop_absorbed=True)

    print(f"\n{'='*80}")
    print(f"  {base_name} → {ext_name}")
    print(f"{'='*80}")

    header = f"{'BW':>4} | {'β3 (dd_model)':>14} {'p':>8} | {'β4 (dd_exp_model)':>18} {'p':>8} | {'β3+β4':>8} {'Wald p':>8}"
    print(header)
    print("-" * len(header))

    for bw in BANDWIDTHS:
        res = mod.fit(cov_type='kernel', kernel='bartlett', bandwidth=bw)

        b3  = res.params['dd_model']
        p3  = res.pvalues['dd_model']
        b4  = res.params['dd_exp_model']
        p4  = res.pvalues['dd_exp_model']
        wald = res.wald_test(formula='dd_model + dd_exp_model = 0')
        wp  = wald.pval

        sig3 = '***' if p3 < 0.01 else ('**' if p3 < 0.05 else ('*' if p3 < 0.1 else ''))
        sig4 = '***' if p4 < 0.01 else ('**' if p4 < 0.05 else ('*' if p4 < 0.1 else ''))
        sigw = '***' if wp < 0.01 else ('**' if wp < 0.05 else ('*' if wp < 0.1 else ''))

        print(f"{bw:>4} | {b3:>14.4f} {p3:>7.4f}{sig3:<3} | {b4:>18.4f} {p4:>7.4f}{sig4:<3} | {b3+b4:>8.4f} {wp:>7.4f}{sigw}")


  M0 → M1
  BW |  β3 (dd_model)        p |  β4 (dd_exp_model)        p |    β3+β4   Wald p
--------------------------------------------------------------------------------
   8 |        -0.0022  0.0075*** |            -0.0087  0.0006*** |  -0.0109  0.0000***
  26 |        -0.0022  0.0946*   |            -0.0087  0.0255**  |  -0.0109  0.0019***
  52 |        -0.0022  0.1905    |            -0.0087  0.0567*   |  -0.0109  0.0080***

  M0 → M2
  BW |  β3 (dd_model)        p |  β4 (dd_exp_model)        p |    β3+β4   Wald p
--------------------------------------------------------------------------------
   8 |        -0.0131  0.0016*** |            -0.0280  0.0000*** |  -0.0412  0.0000***
  26 |        -0.0131  0.0439**  |            -0.0280  0.0030*** |  -0.0412  0.0000***
  52 |        -0.0131  0.0871*   |            -0.0280  0.0138**  |  -0.0412  0.0003***


## Permutation test

In [56]:
np.random.seed(42)
n_perms = 500

placebo_p = {('M0','M1'): [], ('M0','M2'): []}

for base_name, ext_name, base_col, ext_col in comparisons:
    d0 = panel[[COUNTRY_COL, DATE_COL, CDS_COL, base_col, ext_col]].dropna().copy()
    d0 = d0[d0[CDS_COL] > 0].copy()

    for i in range(n_perms):
        # Randomly assign which DD is "model=1" for each row
        swap = np.random.binomial(1, 0.5, size=len(d0)).astype(bool)
        dd_base = np.where(swap, d0[ext_col], d0[base_col])
        dd_ext  = np.where(swap, d0[base_col], d0[ext_col])

        s0 = d0[[COUNTRY_COL, DATE_COL, CDS_COL]].copy()
        s0['DD'] = dd_base
        s0['model'] = 0
        s1 = d0[[COUNTRY_COL, DATE_COL, CDS_COL]].copy()
        s1['DD'] = dd_ext
        s1['model'] = 1

        stacked = pd.concat([s0, s1], ignore_index=True)
        stacked['ln_cds']       = np.log(stacked[CDS_COL])
        stacked['exporter']     = stacked[COUNTRY_COL].isin(EXPORTERS).astype(int)
        stacked['dd_exp']       = stacked['DD'] * stacked['exporter']
        stacked['dd_model']     = stacked['DD'] * stacked['model']
        stacked['dd_exp_model'] = stacked['DD'] * stacked['exporter'] * stacked['model']
        stacked = stacked.set_index([COUNTRY_COL, DATE_COL])

        mod = PanelOLS(stacked['ln_cds'], stacked[['DD','dd_exp','dd_model','dd_exp_model']],
                       entity_effects=True, time_effects=True, drop_absorbed=True)
        res = mod.fit(cov_type='kernel', kernel='bartlett', bandwidth=26)
        placebo_p[(base_name, ext_name)].append(res.params['dd_exp_model'])

    actual = -0.0087 if ext_name == 'M1' else -0.0280
    placebo = np.array(placebo_p[(base_name, ext_name)])
    perm_p = np.mean(placebo <= actual)
    print(f"{base_name}→{ext_name}: actual={actual:.4f}, placebo mean={placebo.mean():.4f}, "
          f"placebo std={placebo.std():.4f}, permutation p={perm_p:.4f}")

M0→M1: actual=-0.0087, placebo mean=-0.0001, placebo std=0.0010, permutation p=0.0000
M0→M2: actual=-0.0280, placebo mean=0.0000, placebo std=0.0011, permutation p=0.0000
